# Model effect: fine grain (trip duration)

Each rung adds features that reach the trips only through the conformed dimensions. Rungs are compared with the one below them on the same held-out trips.

This notebook only presents results. Every number below was computed by the pipeline (`make all`) and is read from `results/` or from the warehouse through a read-only connection; nothing here trains a model or recomputes a reported number. Run it with `make notebooks`.

In [ ]:
profile = "full"

In [ ]:
import duckdb
import pandas as pd
from IPython.display import Image, display

from cigd.config import load_config

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
config = load_config(profile)


def table(name: str) -> pd.DataFrame:
    """Read one result table written by the pipeline."""
    return pd.read_csv(config.tables_dir / f"{name}.csv")


def figure(name: str) -> None:
    """Show one figure written by the pipeline."""
    display(Image(filename=str(config.figures_dir / f"{name}.png"), width=900))


def warehouse_query(sql: str) -> pd.DataFrame:
    """Query the warehouse through a read-only connection."""
    with duckdb.connect(str(config.warehouse_path), read_only=True) as connection:
        return connection.execute(sql).df()

## Training windows per test month

Under the primary information set, training stops four months before the test month (TLC release rule).

In [ ]:
table("fine_origins")

## Accuracy

In [ ]:
table("fine_accuracy")

In [ ]:
figure("fine_accuracy")

## Each rung against the one below it

Effect sizes with 95 percent block-bootstrap intervals (blocks are pickup days), Diebold-Mariano tests and Holm-adjusted p-values. A win needs a lower MAE and an adjusted p below 0.05. The recorded-distance run is a sensitivity run and is not part of the win criteria.

In [ ]:
table("fine_comparisons")[
    [
        "run",
        "information_set",
        "model",
        "comparator",
        "mae_model",
        "mae_comparator",
        "percent_change",
        "percent_change_ci_low",
        "percent_change_ci_high",
        "p_value_absolute",
        "p_value_absolute_holm",
        "win",
    ]
]

In [ ]:
figure("fine_effects")